# Trực quan các approach

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt

## Helper functions — đọc log train

In [ ]:
def _get_metric_records(log_path: Path, metric_key: str) -> list[dict]:
    """
    - Summary: Đọc log train, lọc record có chứa metric_key.
    - Args:
        - log_path: Đường dẫn file JSONL log train.
        - metric_key: Tên field cần lọc (VD: "loss", "mean_token_accuracy").
    - Output:
        - list[dict]: List record có chứa metric_key.
    """
    records: list[dict] = []
    with log_path.open(encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            record = json.loads(line)
            if metric_key in record:  # bỏ dòng summary cuối, không có đủ metric per-step
                records.append(record)
    return records

## Biểu đồ loss trong quá trình train

In [ ]:
def plot_train_loss(loss_log_path: Path | str):
    """
    - Summary: Vẽ biểu đồ loss theo step từ file log train.
    - Args:
        - loss_log_path: Đường dẫn (Path hoặc str) tới file train_loss.jsonl.
    - Output:
        - None. Hiển thị biểu đồ matplotlib.
    """
    log_path = Path(loss_log_path)
    records  = _get_metric_records(log_path, "loss")

    steps  = [record["step"] for record in records]
    losses = [record["loss"] for record in records]

    fig, ax = plt.subplots(figsize=(9, 5))
    ax.plot(steps, losses, linewidth=2, color="#2a78d6")
    ax.set_yscale("log")  # loss trải dài nhiều bậc độ lớn, log scale mới thấy rõ xu hướng hội tụ
    ax.set_xlabel("Step")
    ax.set_ylabel("Loss (log scale)")
    ax.set_title(f"Loss trong quá trình train — {log_path.parent.name}")
    ax.grid(True, which="both", color="#e1e0d9", linestyle="--", linewidth=0.5)
    fig.tight_layout()
    plt.show()

In [ ]:
plot_train_loss("D:/UIT/SE365_Prj/notebooks/v1_augmented_train_loss.jsonl")

## Biểu đồ mean_token_accuracy trong quá trình train

In [ ]:
def plot_train_token_accuracy(loss_log_path: Path | str):
    """
    - Summary: Vẽ biểu đồ mean_token_accuracy theo step từ file log train.
    - Args:
        - loss_log_path: Đường dẫn (Path hoặc str) tới file train_loss.jsonl.
    - Output:
        - None. Hiển thị biểu đồ matplotlib.
    """
    log_path = Path(loss_log_path)
    records  = _get_metric_records(log_path, "mean_token_accuracy")

    steps      = [record["step"] for record in records]
    accuracies = [record["mean_token_accuracy"] for record in records]

    fig, ax = plt.subplots(figsize=(9, 5))
    ax.plot(steps, accuracies, linewidth=2, color="#1baf7a")  # slot 2 (aqua) — phân biệt với plot_train_loss
    ax.set_ylim(0, 1.02)
    ax.set_xlabel("Step")
    ax.set_ylabel("Mean token accuracy")
    ax.set_title(f"Mean token accuracy trong quá trình train — {log_path.parent.name}")
    ax.grid(True, color="#e1e0d9", linestyle="--", linewidth=0.5)
    fig.tight_layout()
    plt.show()

In [ ]:
plot_train_token_accuracy("D:/UIT/SE365_Prj/ml/approaches/qwen_7b_instruct/output/train_v1/train_loss.jsonl")